# 멀티모달 RAG: 텍스트 · 표 · 이미지를 함께 검색하고 답하기

> 파이프라인 3단계: **① 추출 → ② 요약 → ③ 요약으로 검색 + 원본으로 생성**
>
> 사용 전 `TODO` 표시를 모두 본인 자료에 맞게 바꾸세요. 맨 마지막에 **Restart & Run All**로 출력 셀을 남기고 저장해야 합니다.

| 단계 | 하는 일 | 저장 위치 |
|---|---|---|
| ① 추출 | PDF에서 텍스트 · 표 · 이미지를 분리 | `figures/`, 메모리 |
| ② 요약 | 각 요소를 검색용 요약문으로 변환 | `cache/summaries.json` |
| ③ 검색·생성 | 요약으로 검색 → 연결된 **원본**을 모델에 전달 | `docs/qa_results.md` |

## 0. 설정

In [ ]:
# 처음 한 번만 실행 (이미 설치돼 있으면 건너뛰세요)
# %pip install -q pymupdf pdfplumber openai numpy python-dotenv

In [ ]:
import os, json, base64
from pathlib import Path

import numpy as np
import fitz          # PyMuPDF: 텍스트, 이미지, 페이지 렌더링
import pdfplumber    # 표 추출
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # .env 의 OPENAI_API_KEY 를 환경변수로 읽음 (키는 코드에 쓰지 않음)
assert os.getenv("OPENAI_API_KEY"), ".env 에 OPENAI_API_KEY 를 설정하세요"
client = OpenAI()

# ---------- TODO: 본인 자료에 맞게 수정 ----------
PDF_PATH = "data/my_report.pdf"      # 직접 고른 PDF 경로
PROMPT_LANG = "ko"                   # 요약/답변 프롬프트 언어: "ko" 또는 "en" (선택 근거는 README에)

SUMMARY_MODEL = "gpt-4o-mini"        # 요약 + 이미지 설명 (비전 지원 모델)
ANSWER_MODEL = "gpt-4o-mini"         # 최종 답변 (비전 지원 모델)
EMBED_MODEL = "text-embedding-3-small"

CHUNK_SIZE = 1500                    # 텍스트 청크 글자 수 (선택 근거는 README에)
CHUNK_OVERLAP = 150
TOP_K = 4                            # 검색 시 가져올 요소 수

MIN_IMG_SIZE = 150                   # 이보다 작은 이미지(로고, 아이콘)는 제외
VECTOR_THRESHOLD = 40                # 벡터 그래프로 추정할 페이지의 도형 수
FORCE_RENDER_PAGES = []              # 그래프가 있는데 자동 감지가 안 되면 페이지 번호를 직접 추가
# ------------------------------------------------

FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)
DOCS_DIR = Path("docs"); DOCS_DIR.mkdir(exist_ok=True)
assert Path(PDF_PATH).exists(), f"PDF 가 없습니다: {PDF_PATH}"
print("설정 완료:", PDF_PATH)

## 1단계. 데이터 추출

- **텍스트**: 페이지별로 뽑아 일정 길이로 자름
- **표**: `pdfplumber`로 뽑아 마크다운 표로 변환
- **이미지**: 삽입된 이미지는 그대로 저장하고, 벡터로 그려진 그래프는 페이지를 이미지로 렌더링해 저장

In [ ]:
def chunk_text(text, size, overlap):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


def table_to_markdown(rows):
    rows = [[(c or "").replace("\n", " ").strip() for c in r] for r in rows]
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    header, body = rows[0], rows[1:]
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in body]
    return "\n".join(lines)


elements = []   # 모든 원본 요소: {id, type, page, content}


# ---- 텍스트 + 이미지 (PyMuPDF) ----
doc = fitz.open(PDF_PATH)
for pno, page in enumerate(doc, start=1):
    text = page.get_text("text").strip()
    for i, ch in enumerate(chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)):
        elements.append({"id": f"text-p{pno}-{i}", "type": "text", "page": pno, "content": ch})

    saved_embedded = 0
    for j, img in enumerate(page.get_images(full=True)):
        pix = fitz.Pixmap(doc, img[0])
        if pix.n - pix.alpha >= 4:                  # CMYK 등은 RGB 로 변환
            pix = fitz.Pixmap(fitz.csRGB, pix)
        if pix.width < MIN_IMG_SIZE or pix.height < MIN_IMG_SIZE:
            continue
        path = FIG_DIR / f"p{pno}_img{j}.png"
        pix.save(path)
        saved_embedded += 1
        elements.append({"id": f"image-p{pno}-{j}", "type": "image", "page": pno, "content": str(path)})

    # 삽입 이미지가 없는데 도형이 많으면 벡터 그래프로 보고 페이지를 통째로 렌더링
    is_vector_chart = saved_embedded == 0 and len(page.get_drawings()) >= VECTOR_THRESHOLD
    if is_vector_chart or pno in FORCE_RENDER_PAGES:
        path = FIG_DIR / f"p{pno}_page.png"
        page.get_pixmap(dpi=130).save(path)
        elements.append({"id": f"image-p{pno}-page", "type": "image", "page": pno, "content": str(path)})


# ---- 표 (pdfplumber) ----
with pdfplumber.open(PDF_PATH) as pdf:
    for pno, page in enumerate(pdf.pages, start=1):
        for k, tbl in enumerate(page.extract_tables()):
            if not tbl or len(tbl) < 2:
                continue
            elements.append({"id": f"table-p{pno}-{k}", "type": "table", "page": pno,
                             "content": table_to_markdown(tbl)})

docstore = {e["id"]: e for e in elements}   # id -> 원본 (생성 단계에서 사용)

from collections import Counter
print("추출 결과:", dict(Counter(e["type"] for e in elements)))
print("페이지 수:", len(doc))

In [ ]:
# 추출 결과 확인: 표 1개, 이미지 1개 미리보기
from IPython.display import Image, Markdown, display

tables = [e for e in elements if e["type"] == "table"]
images = [e for e in elements if e["type"] == "image"]

if tables:
    print(f"[표 예시] {tables[0]['id']}")
    display(Markdown(tables[0]["content"]))
if images:
    print(f"[이미지 예시] {images[0]['id']}")
    display(Image(filename=images[0]["content"], width=500))
if len(tables) + len(images) < 2:
    print("경고: 표/그래프가 2개 미만으로 추출됐습니다. VECTOR_THRESHOLD 또는 FORCE_RENDER_PAGES 를 조정하세요.")

## 2단계. 요약 생성

원본을 그대로 임베딩하면 표의 숫자나 그래프 이미지는 검색이 잘 안 됩니다.
그래서 **각 요소를 검색용 요약문(텍스트)으로 바꾸고**, 원본은 따로 보관합니다.
요약 호출은 비용이 들기 때문에 결과를 `cache/summaries.json`에 저장해 재실행 시 재사용합니다.

In [ ]:
PROMPTS = {
    "ko": {
        "text": "다음 텍스트를 검색에 쓸 수 있게 3문장 이내로 요약하세요. 핵심 주제, 수치, 고유명사를 포함하세요.\n\n{content}",
        "table": "다음 표가 무엇을 나타내는지 3문장 이내로 요약하세요. 표 제목, 항목, 핵심 수치와 추세를 포함하세요.\n\n{content}",
        "image": "이 이미지(그래프/도표/사진)를 검색에 쓸 수 있게 설명하세요. 종류, 제목, 축과 단위, 핵심 수치와 추세를 3~4문장으로 쓰세요.",
        "system": "당신은 문서 내용을 정확하게 요약하는 조수입니다. 없는 내용은 지어내지 마세요.",
    },
    "en": {
        "text": "Summarize the following text in at most 3 sentences for retrieval. Keep key topics, figures and proper nouns.\n\n{content}",
        "table": "Summarize what this table shows in at most 3 sentences. Include title, items, key figures and trends.\n\n{content}",
        "image": "Describe this image (chart/diagram/photo) for retrieval: type, title, axes and units, key figures and trends, in 3-4 sentences.",
        "system": "You summarize document content accurately. Never invent content.",
    },
}[PROMPT_LANG]


def encode_image(path):
    return base64.b64encode(Path(path).read_bytes()).decode()


def chat(messages, model):
    r = client.chat.completions.create(model=model, messages=messages, temperature=0)
    return r.choices[0].message.content.strip()


def summarize(e):
    if e["type"] == "image":
        content = [
            {"type": "text", "text": PROMPTS["image"]},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encode_image(e['content'])}"}},
        ]
    else:
        content = PROMPTS[e["type"]].format(content=e["content"])
    return chat([{"role": "system", "content": PROMPTS["system"]},
                 {"role": "user", "content": content}], SUMMARY_MODEL)


SUMMARY_PATH = CACHE_DIR / "summaries.json"
summaries = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else {}

for e in elements:
    if e["id"] in summaries:
        continue
    summaries[e["id"]] = summarize(e)
    SUMMARY_PATH.write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"요약 {len(summaries)}개 생성/로드 완료")

In [ ]:
# 요약 품질 확인: 유형별 1개씩 (원본 vs 요약)
for t in ["text", "table", "image"]:
    e = next((x for x in elements if x["type"] == t), None)
    if e:
        original = e["content"] if t != "image" else f"(이미지 파일) {e['content']}"
        print(f"=== [{t}] {e['id']} ===")
        print("원본:", original[:200].replace("\n", " "))
        print("요약:", summaries[e["id"]], "\n")

## 3단계. 요약으로 검색 → 원본으로 생성

- **검색**: 질문을 임베딩해서 *요약문*과 비교 (요약이 의미 검색에 유리)
- **생성**: 검색된 id로 *원본*(텍스트, 표, 이미지)을 꺼내 모델에 전달 (원본이 정확한 수치를 담고 있음)

In [ ]:
def embed(texts):
    vecs = []
    for i in range(0, len(texts), 100):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 100])
        vecs += [d.embedding for d in r.data]
    m = np.array(vecs, dtype=np.float32)
    return m / np.linalg.norm(m, axis=1, keepdims=True)


class VectorIndex:
    def __init__(self, ids, texts):
        self.ids = ids
        self.matrix = embed(texts)

    def search(self, query, k):
        q = embed([query])[0]
        scores = self.matrix @ q
        top = np.argsort(-scores)[:k]
        return [(self.ids[i], float(scores[i])) for i in top]


# 멀티모달 인덱스: 모든 요소의 '요약'을 임베딩
mm_ids = [e["id"] for e in elements]
mm_index = VectorIndex(mm_ids, [summaries[i] for i in mm_ids])

# 비교용 텍스트 RAG 인덱스: 텍스트 원문 청크만 임베딩 (표/이미지 정보 없음)
text_elems = [e for e in elements if e["type"] == "text"]
text_index = VectorIndex([e["id"] for e in text_elems], [e["content"] for e in text_elems])
print("인덱스 생성 완료:", len(mm_ids), "개(멀티모달),", len(text_elems), "개(텍스트 전용)")

In [ ]:
ANSWER_SYSTEM = {
    "ko": "주어진 자료만 근거로 한국어로 답하세요. 자료에 없으면 '자료에서 확인할 수 없습니다'라고 답하세요. "
          "답변 끝에 근거로 쓴 자료를 (종류, 페이지) 형식으로 적으세요.",
    "en": "Answer only from the provided context. If it is not there, say you cannot find it. "
          "End with the sources you used as (type, page).",
}[PROMPT_LANG]


def build_context(ids):
    parts = []
    for i in ids:
        e = docstore[i]
        if e["type"] == "image":
            parts.append({"type": "text", "text": f"[image | p.{e['page']}]"})
            parts.append({"type": "image_url",
                          "image_url": {"url": f"data:image/png;base64,{encode_image(e['content'])}"}})
        else:
            parts.append({"type": "text", "text": f"[{e['type']} | p.{e['page']}]\n{e['content']}"})
    return parts


def answer(question, index, k=TOP_K):
    hits = index.search(question, k)
    ids = [h[0] for h in hits]
    content = build_context(ids) + [{"type": "text", "text": f"질문: {question}"}]
    reply = chat([{"role": "system", "content": ANSWER_SYSTEM},
                  {"role": "user", "content": content}], ANSWER_MODEL)
    return {"answer": reply, "ids": ids, "types": sorted({docstore[i]["type"] for i in ids})}


def ask_multimodal(q):
    return answer(q, mm_index)


def ask_text_only(q):
    return answer(q, text_index)   # 텍스트 청크만 검색하므로 표/그래프 근거는 쓸 수 없음

## 질의응답 5건

> **TODO**: 본인 PDF를 읽고 질문을 직접 작성하세요. 텍스트, 표, 그래프가 각각 근거가 되는 질문을 섞어야
> 텍스트 RAG와의 차이가 드러납니다. `expected` 는 정답이 있는 곳을 미리 적어 두는 칸입니다(채점 근거 확인용).

In [ ]:
QUESTIONS = [
    {"q": "TODO: 본문 텍스트에서 답이 나오는 질문",  "expected": "text"},
    {"q": "TODO: 표의 수치를 묻는 질문",            "expected": "table"},
    {"q": "TODO: 그래프의 추세나 값을 묻는 질문",      "expected": "image"},
    {"q": "TODO: 표와 텍스트를 함께 봐야 하는 질문",   "expected": "table+text"},
    {"q": "TODO: 그래프와 텍스트를 함께 봐야 하는 질문", "expected": "image+text"},
]

results = []
for item in QUESTIONS:
    mm = ask_multimodal(item["q"])
    tx = ask_text_only(item["q"])
    results.append({**item, "mm": mm, "tx": tx})
    print("Q:", item["q"])
    print("  [멀티모달]", mm["answer"][:300].replace("\n", " "))
    print("  [텍스트만]", tx["answer"][:300].replace("\n", " "))
    print("  검색된 근거 종류:", mm["types"], "\n")

## Before / After 비교 (텍스트 RAG vs 멀티모달 RAG)

같은 질문에 두 방식이 어떻게 다르게 답했는지 한 표로 보여 줍니다.
이 셀의 출력을 **README에 캡처**하세요.

In [ ]:
from IPython.display import Markdown, display

def cell(s):
    return s.replace("|", "\\|").replace("\n", " ")

rows = ["| # | 질문 | Before: 텍스트 RAG | After: 멀티모달 RAG | 근거가 된 원본 종류 |",
        "|---|---|---|---|---|"]
for n, r in enumerate(results, start=1):
    rows.append(f"| {n} | {cell(r['q'])} | {cell(r['tx']['answer'])} | {cell(r['mm']['answer'])} | {', '.join(r['mm']['types'])} |")

table_md = "\n".join(rows)
display(Markdown(table_md))
(DOCS_DIR / "qa_results.md").write_text(table_md, encoding="utf-8")
print("저장: docs/qa_results.md")

## 정리 (README에 본인 말로 쓸 내용)

아래 질문에 직접 답을 적어 두면 README 작성이 빨라집니다. **복사하지 않고 본인 말로** 써야 합니다.

- 왜 **요약으로 검색**하고 **원본으로 생성**했나요? 원본을 바로 임베딩하면 무엇이 문제였나요?
- 모델(`SUMMARY_MODEL`, `ANSWER_MODEL`, `EMBED_MODEL`)을 왜 골랐나요?
- 청크 크기(`CHUNK_SIZE`)를 왜 그렇게 정했나요? 바꿔 보니 어땠나요?
- 프롬프트 언어(`PROMPT_LANG`)를 왜 골랐나요?
- 실행하면서 실제로 겪은 한계는 무엇인가요? (표가 깨짐, 그래프 해석 오류, 비용 등)